### ЗАДАЧА: Учёт посещаемости мероприятий

Есть список регистраций участников на внутренние мероприятия.
Нужно обработать его через классы и собрать отчёт по событиям и участникам.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Класс `AttendanceRecord` с полями `person`, `event_name`, `day`, `status`.

2. В `AttendanceRecord` реализовать:
   - метод `to_dict()`
   - `@classmethod from_tuple(cls, row)`.

3. В `AttendanceRecord.from_tuple(cls, row)`:
   - если формат строки неверный, вернуть `(None, "bad format")`
   - если статус не входит в допустимые, вернуть `(None, "bad status")`
   - если запись корректна, вернуть `(record, "")`.

4. Класс `AttendanceRegistry`:
   - хранит записи в списке `self.records`
   - хранит невалидные строки в списке `self.invalid_rows`
   - хранит дубли в списке `self.duplicates`.

5. В `AttendanceRegistry` реализовать методы:
   - `add_record(record)`
   - `load_rows(rows)`
   - `event_stats()`
   - `person_history(person)`
   - `active_events()`
   - `build_report()`.

6. При добавлении записи:
   - считать дублем совпадение `person`, `event_name`, `day`
   - дубль не добавлять
   - информацию о дубле сохранять в `self.duplicates`.

7. После загрузки данных вывести:
   - количество корректных записей
   - количество ошибок
   - количество дублей
   - статистику по мероприятиям
   - множество активных мероприятий.


In [ ]:
rows = [
    ("Алиса", "Python Meetup", "Mon", "registered"),
    ("Алиса", "Python Meetup", "Mon", "attended"),
    ("Боб", "Data Talk", "Tue", "registered"),
    ("Кира", "Data Talk", "Tue", "attended"),
    ("Данил", "Design Review", "Wed", "cancelled"),
    ("Ева", "Python Meetup", "Thu", "waiting"),
    ("Боб", "Data Talk", "Tue", "registered")
]


class AttendanceRecord:
    # TODO:
    # реализуй __init__(person, event_name, day, status)
    def __init__(self, person, event_name, day, status):
        self.person = person
        self.event_name = event_name
        self.day = day
        self.status = status

    def to_dict(self):
        # TODO:
        # вернуть словарь с полями person, event_name, day, status
        return {'person': self.person, 'event_name': self.event_name, 'day': self.day, 'status': self.status}
        

    @classmethod
    def from_tuple(cls, row):
        # TODO:
        # 1) проверить длину row
        # 2) распаковать значения
        # 3) проверить статус
        # 4) вернуть (record, "") или (None, reason)
        if len(row) != 4:
            return (None, 'Bad format')
        if row[-1] not in ["registered", "attended", "cancelled", "waiting"]:
            return (None, 'Bad status')
        return (cls(*row), '')



class AttendanceRegistry:
    def __init__(self):
        # TODO:
        self.records = []
        self.invalid_rows = []
        self.duplicates = []
        

    def add_record(self, record):
        # TODO:
        # проверить, есть ли уже запись с теми же person, event_name, day
        # если да:
        #    добавить (record.person, record.event_name, record.day) в self.duplicates
        #    не добавлять запись
        # иначе добавить record в self.records
        for el in self.records:
            if el.person == record.person and el.event_name == record.event_name and el.day == record.day:
                self.duplicates.append((record.person, record.event_name, record.day))
                return
        self.records.append(record)

    def load_rows(self, rows):
        # TODO:
        # пройти по rows
        # вызвать AttendanceRecord.from_tuple(row)
        # если запись невалидна -> self.invalid_rows.append((row, reason))
        # иначе -> self.add_record(record)
        for row in rows:
            tr = AttendanceRecord.from_tuple(row)
            if tr[0] == None:
                self.invalid_rows.append((row, tr[1]))
            else:
                self.add_record(tr[0])

    def event_stats(self):
        # TODO:
        # вернуть словарь статистики по мероприятиям и статусам
        obj = {}
        for el in self.records:
            obj.setdefault(el.event_name, [])
            obj[el.event_name].append(el.status)
        return obj


    def person_history(self, person):
        # TODO:
        # вернуть список записей выбранного участника
        for el in self.records:
            if el.person == person:
                return((el.event_name, el.day, el.status))
        

    def active_events(self):
        # TODO:
        # вернуть множество событий, где есть хотя бы один status == 'attended'
        s = set()
        for el in self.records:
            if el.status == 'attended':
                s.add(el.event_name)

    def build_report(self):
        # TODO:
        # вернуть словарь с ключами:
        # total_valid_records, total_invalid_rows,
        # total_duplicates, event_stats, active_events
        obj = {'total_valid_records': None, 'total_invalid_rows': None,
        'total_duplicates': None, 'event_stats':None, 'active_events':None}
        obj['total_valid_records'] = len(self.records)
        obj['total_invalid_rows'] = len(self.invalid_rows)
        obj['total_duplicates'] = len(self.duplicates)
        obj['event_stats'] = self.event_stats()
        obj['active_events'] = self.active_events()

registry = AttendanceRegistry()
registry.load_rows(rows)
report = registry.build_report()

print("Корректных записей:", report["total_valid_records"])
print("Ошибок:", report["total_invalid_rows"])
print("Дублей:", report["total_duplicates"])
print("Статистика мероприятий:", report["event_stats"])
print("Активные мероприятия:", report["active_events"])
